# Turn chunks into embeddings

Tutorial 50 made chunks. Here an embedding model counts its real tokens and turns a few real chunks into equal-length lists of numbers. Nothing is searched or ranked yet; tutorial 70 does that.

This tutorial is standalone. First run the setup command in the README and select QAChat's `.venv` kernel.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Python environment: {sys.executable}")


## 1. Load an embedding model

`auto` selects MLX on Apple Silicon and PyTorch elsewhere. Each backend uses compatible weights from Hugging Face's normal cache.


In [ ]:
from qachat.runtime import choose_backend

BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]
print(selected_backend, MODEL_ID)


In [ ]:
from qachat.embeddings import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare real repository chunks

The lower steps are repeated so this tutorial can run by itself. It reads the same real QAChat file used in tutorial 50.


In [ ]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat/repository.py"],
)

print(report["selected_files"][0])


In [ ]:
from qachat.source import extract_source_units
from qachat.source import read_repository_files

source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Natural units: {len(units)}")


In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=120,
    max_tokens=160,
    overlap_tokens=20,
)

print(f"Chunks: {len(chunks)}")


## 3. Compare words with model tokens

A token can be a word, part of a word, punctuation, or a code mark. The model's token limit—not the visible word count—decides how much text fits.


In [ ]:
first_chunk = chunks[0]
word_count = len(first_chunk.text.split())
token_count = embedding_session.count_tokens(first_chunk.text)

print(f"Visible words: {word_count}")
print(f"Model tokens: {token_count}")


## 4. Turn chunks into vectors

One chunk becomes one embedding vector. Every vector from the same model has the same length. Its values are learned numeric features, not readable words.


In [ ]:
selected_chunks = chunks[:3]
texts = [chunk.text for chunk in selected_chunks]
document_vectors = embedding_session.embed_documents(texts)

print(f"Chunks sent: {len(selected_chunks)}")
print(f"Vectors returned: {len(document_vectors)}")


In [ ]:
dimensions = {len(vector) for vector in document_vectors}
assert len(document_vectors) == len(selected_chunks)
assert len(dimensions) == 1

dimension = dimensions.pop()
assert dimension > 0
print(f"Vector length: {dimension}")


In [ ]:
for chunk, vector in zip(
    selected_chunks,
    document_vectors,
    strict=True,
):
    name = chunk.symbol or "(module context)"
    print(name, vector[:5])


## 5. Turn a question into a vector

A question receives a vector in the same numeric space as the chunks. This prepares it for a later comparison but does not search anything.


In [ ]:
QUESTION = "How does QAChat choose repository files?"
question_vector = embedding_session.embed_query(QUESTION)

assert len(question_vector) == dimension
print(f"Question vector length: {len(question_vector)}")
print(question_vector[:5])


## 6. Release the model


In [ ]:
embedding_session.close()
print("Embedding model released.")


## Result

The chunks and the question became vectors of the same length. No index was built and no result was retrieved. Tutorial 70 compares these vectors to search by meaning.
